In [1]:
!pip install -q -U kaggle

In [2]:
import os
from google.colab import userdata

os.environ["KAGGLE_API_TOKEN"] = userdata.get("KAGGLE_API_TOKEN")

In [3]:
!kaggle datasets download -d ealaxi/paysim1 -p /content/data
!unzip -o -q /content/data/paysim1.zip -d /content/data
!ls -lh /content/data

Dataset URL: https://www.kaggle.com/datasets/ealaxi/paysim1
License(s): CC-BY-SA-4.0
100% 178M/178M [00:01<00:00, 157MB/s]

total 649M
-rw-r--r-- 1 root root 178M Sep 20  2019 paysim1.zip
-rw-r--r-- 1 root root 471M Sep 20  2019 PS_20174392719_1491204439457_log.csv


In [4]:
import hashlib

path = "/content/data/PS_20174392719_1491204439457_log.csv"
h = hashlib.sha256()
with open(path, "rb") as f:
    for chunk in iter(lambda: f.read(1 << 20), b""):
        h.update(chunk)
print(h.hexdigest())

16910f90577b0d981bf8ff289714510bb89bc71bff7d3f220f024e287e4eea6b


In [5]:
import pandas as pd

df = pd.read_csv(path)

print(df.shape)
print(df.isFraud.sum(), round(df.isFraud.mean() * 100, 3))
print(df.isnull().sum().sum())
print(df.duplicated().sum())
print(df.nameOrig.nunique(), df.nameDest.nunique())
print(df.step.min(), df.step.max())
print(df[df.isFraud == 1].type.value_counts())

steps = sorted(df.step.unique())
n = len(steps)
s1, s2 = steps[int(n * 0.6) - 1], steps[int(n * 0.8) - 1]
part = pd.cut(df.step, [0, s1, s2, df.step.max()], labels=["train", "val", "test"])
print(s1, s2)
print(df.groupby(part, observed=True).isFraud.agg(["count", "sum", "mean"]))

(6362620, 11)
8213 0.129
0
0
6353307 2722362
1 743
type
CASH_OUT    4116
TRANSFER    4097
Name: count, dtype: int64
445 594
         count   sum      mean
step                          
train  6010937  5007  0.000833
val     228103  1552  0.006804
test    123580  1654  0.013384


In [6]:
!pip freeze | grep -iE "^(pandas|numpy|scikit-learn|xgboost|matplotlib|kaggle)==" > requirements.txt
!cat requirements.txt

from google.colab import files
files.download("requirements.txt")

kaggle==2.2.5
matplotlib==3.10.0
numpy==2.4.6
pandas==2.2.3
scikit-learn==1.6.1
xgboost==3.4.1


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>